### What it does
Script to handle words with "-", for example "risk-taker". The aim is to modify it into "risk taker", "risktaker". 
It searchs "-" in words ignoring case. Then it split the words by "-" and joint the words in one taking into account lowcase letter.
It creates new table with the the same columns as original table. Original table with new one are merged and sorted by lemma. 

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# pandas library to work with tables/dataframes
import pandas as pd

# Regular expression library
import re
import ast

In [3]:
#file name
file_name = "rules_29JUN_German"

In [4]:
df_rules = pd.read_csv(file_name+".csv")

In [5]:
#check first 5 rows
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,...,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives,Sample_Sentences,False_Positives,Rule copy,3rd_party_source,ToClarify,Notes
0,dämlich,a,26-01-2022,gendered,female_stereotype,NaN,NaN,ungünstig | dumm | ungeschickt | unbeholfen | ...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Frauenzimmer,s,26-01-2022,gendered,female_stereotype,NaN,NaN,Frau | Kollegin | Teammitglied | Fachbereichsm...,NaN,NaN,...,Mitarbeitende,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,jammern,v,5/1/2022,gendered,female_stereotype,culture,NaN,kritisieren,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Lehrtochter,s,NaN,gendered,female_stereotype,NaN,Ausbildung | Lehre,Nachwuchskraft | Azubi,Auszubildend~e~/~Auszubildender,Person in Ausbildung | auszubildende Person | ...,...,Lernende | Auszubildende,NaN,Personen in Ausbildung | auszubildende Persone...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,nörgeln,v,NaN,gendered,female_stereotype,NaN,NaN,motivieren | kommentieren | sich beschweren | ...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Remove unnecessary columns

In [6]:
#create a subset of dataframe from the 'Lemma' till the last column with alternatives '3rd_party_alternatives'
df_rules=df_rules.loc[:, 'Lemma':'3rd_party_alternatives']

In [7]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2135 entries, 0 to 2134
Data columns (total 16 columns):
Lemma                             2134 non-null object
Word_Type                         2133 non-null object
Priority                          230 non-null object
Category                          2135 non-null object
Primary_subcategory               2135 non-null object
Secondary_subcategory             797 non-null object
Alt_Field                         854 non-null object
Alt_Sg_Replacement                937 non-null object
Alt_Sg_/_and_inclusive_form       280 non-null object
Alt_Sg_people_first               224 non-null object
Alt_Sg_reframed                   210 non-null object
Alt_Pl_collective_noun            46 non-null object
Alt_Pl                            165 non-null object
Alt_Pl_pair_and_inclusive_form    241 non-null object
Alt_Pl_people_first               122 non-null object
3rd_party_alternatives            1 non-null object
dtypes: object(16)
memory us

In [8]:
#check the number of columns
len(list(df_rules.columns))

16

In [9]:
# delete rows with empty values in "Lemma", "Word_Type", 'Category', "Primary_subcategory"  
df_rules = df_rules.dropna(subset = ["Lemma", "Word_Type", 'Category', "Primary_subcategory"])

In [10]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2132 entries, 0 to 2134
Data columns (total 16 columns):
Lemma                             2132 non-null object
Word_Type                         2132 non-null object
Priority                          229 non-null object
Category                          2132 non-null object
Primary_subcategory               2132 non-null object
Secondary_subcategory             796 non-null object
Alt_Field                         854 non-null object
Alt_Sg_Replacement                936 non-null object
Alt_Sg_/_and_inclusive_form       280 non-null object
Alt_Sg_people_first               223 non-null object
Alt_Sg_reframed                   209 non-null object
Alt_Pl_collective_noun            46 non-null object
Alt_Pl                            165 non-null object
Alt_Pl_pair_and_inclusive_form    241 non-null object
Alt_Pl_people_first               122 non-null object
3rd_party_alternatives            1 non-null object
dtypes: object(16)
memory us

### Remove white spaces from Category and Primary_subcategory

In [11]:
df_rules['Primary_subcategory'] = df_rules['Primary_subcategory'].str.strip()

In [12]:
df_rules['Category'] = df_rules['Category'].str.strip()

### Convert word_type to lowcase

In [13]:
df_rules['Word_Type'] = df_rules['Word_Type'].str.lower()

### Splitting and merging words by hyphen

In [14]:
list_word = []

list_word_type = []
list_priority = []
list_category = []
list_primary_subcategory = []
list_secondary_subcategory = []
list_alt_field= []
list_alt_sg_replacement = []
list_alt_sg_and_inclusive_form = []
list_alt_sg_people_first = []
list_alt_sg_reframed = []
list_alt_pl_collective_noun = []
list_alt_pl = []
list_alt_pl_pair_and_inclusive_form = []
list_alt_pl_people_first = []
list_notes = []

list_3rd_party_alternatives = []
list_3rd_party_source = []
list_toclarify = []

for index, row in df_rules.iterrows():
    word = row["Lemma"]
    span = re.search("-", word)
    #filter words with only one "-"
    if type(span) == re.Match and len(word.rsplit("-"))==2:
        #merge the words by "-" and save the results in lists
        list_word.append(word.rsplit("-")[0]+word.rsplit("-")[1].lower())
        
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_and_inclusive_form.append(row["Alt_Sg_/_and_inclusive_form"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_pair_and_inclusive_form.append(row["Alt_Pl_pair_and_inclusive_form"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        
        #split the words by "-", and then save it to the list
        list_word.append(word.rsplit("-")[0]+" "+word.rsplit("-")[1])
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_and_inclusive_form.append(row["Alt_Sg_/_and_inclusive_form"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_pair_and_inclusive_form.append(row["Alt_Pl_pair_and_inclusive_form"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])  
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        

### New created dataframe

In [15]:
data = pd.DataFrame(list(zip(list_word,list_word_type,list_priority, list_category, list_primary_subcategory, list_secondary_subcategory, 
list_alt_field, list_alt_sg_replacement, list_alt_sg_and_inclusive_form, list_alt_sg_people_first, list_alt_sg_reframed, list_alt_pl_collective_noun,
list_alt_pl, list_alt_pl_pair_and_inclusive_form, list_alt_pl_people_first, list_3rd_party_alternatives)), columns=list(df_rules.columns))

In [16]:
# check first 5 rows of new dataframe
data.head(5)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
0,Corporateinfluencer,s,05-02-2022,gendered,function,americanism,Corporate Influencing,NaN,Corporate-Influencer~in~/~Corporate-Influencer...,"Person, die ihre Marke mit Energie und Ideen a...",NaN,NaN,NaN,Corporate-Influencer~innen~ und ~Corporate-Inf...,"Mitarbeitende, die im Netz rund um Themen ihre...",NaN
1,Corporate Influencer,s,05-02-2022,gendered,function,americanism,Corporate Influencing,NaN,Corporate-Influencer~in~/~Corporate-Influencer...,"Person, die ihre Marke mit Energie und Ideen a...",NaN,NaN,NaN,Corporate-Influencer~innen~ und ~Corporate-Inf...,"Mitarbeitende, die im Netz rund um Themen ihre...",NaN
2,Guerillamarketing,s,26-01-2022,gendered,hidden_image,NaN,NaN,kostengünstige unkonventionelle Werbestrategie...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Military terms
3,Guerilla Marketing,s,26-01-2022,gendered,hidden_image,NaN,NaN,kostengünstige unkonventionelle Werbestrategie...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Military terms
4,Backender,s,NaN,gendered,titles,americanism,Back-End-Entwicklung,NaN,Backend-Entwickler~in~/~Backend-Entwickler,"Person, die den technischen Unterbau entwickel...",NaN,NaN,NaN,Backend-Entwickler~innen~ und ~Backend-Entwickler,"Personen, die den technischen Unterbau entwick...",NaN


In [17]:
#general info about new dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 16 columns):
Lemma                             102 non-null object
Word_Type                         102 non-null object
Priority                          48 non-null object
Category                          102 non-null object
Primary_subcategory               102 non-null object
Secondary_subcategory             16 non-null object
Alt_Field                         26 non-null object
Alt_Sg_Replacement                52 non-null object
Alt_Sg_/_and_inclusive_form       14 non-null object
Alt_Sg_people_first               28 non-null object
Alt_Sg_reframed                   4 non-null object
Alt_Pl_collective_noun            2 non-null object
Alt_Pl                            8 non-null object
Alt_Pl_pair_and_inclusive_form    10 non-null object
Alt_Pl_people_first               24 non-null object
3rd_party_alternatives            2 non-null object
dtypes: object(16)
memory usage: 12.9+ KB


### Add new dataframe to existing one

In [18]:
df_fin = df_rules.append(data, ignore_index = True)

In [19]:
#check final dataframe
df_fin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2234 entries, 0 to 2233
Data columns (total 16 columns):
Lemma                             2234 non-null object
Word_Type                         2234 non-null object
Priority                          277 non-null object
Category                          2234 non-null object
Primary_subcategory               2234 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

In [20]:
#check last 5 rows of new dataframe since the new data were added by the end
df_fin.tail()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
2229,D&O Versicherung,abbr,NaN,style,abbreviation,business,NaN,Directors-and-Officers-Versicherung | Organ-Ha...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2230,Kptsb,acr,NaN,style,abbreviation,medical,NaN,Komplexe posttraumatische Belastungsstörung (k...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2231,K PTSB,acr,NaN,style,abbreviation,medical,NaN,Komplexe posttraumatische Belastungsstörung (k...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2232,Pcard,abbr,NaN,style,abbreviation,finance,NaN,Purchasing-Card (P-card) | Procurement-Card (P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2233,P card,abbr,NaN,style,abbreviation,finance,NaN,Purchasing-Card (P-card) | Procurement-Card (P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Sorted alphabeticaly "Lemma"

In [21]:
df_sorted = df_fin.sort_values(by=['Lemma'], inplace=False, ascending=True)

In [22]:
#check sorted dataframe
df_sorted.head(5)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
1755,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service ((x)aaS) | etwas als Di...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1756,ACCT,abbr,NaN,style,abbreviation,finance,NaN,Account | Konto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1757,ADD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Disorder | Attention Deficit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1759,ADHD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Hyperactivity Disorder | Att...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1760,ADHS,acr,NaN,style,abbreviation,medical,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Save to csv file after preprocessing

In [23]:
df_sorted.to_csv(file_name+"_h"+".csv", index=False)

In [24]:
df_test = pd.read_csv(file_name+"_h"+".csv")

In [25]:
df_test.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service ((x)aaS) | etwas als Di...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACCT,abbr,NaN,style,abbreviation,finance,NaN,Account | Konto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ADD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Disorder | Attention Deficit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ADHD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Hyperactivity Disorder | Att...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ADHS,acr,NaN,style,abbreviation,medical,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2235 entries, 0 to 2234
Data columns (total 16 columns):
Lemma                             2234 non-null object
Word_Type                         2234 non-null object
Priority                          277 non-null object
Category                          2234 non-null object
Primary_subcategory               2234 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us